<a href="https://colab.research.google.com/github/GoudoMahan/AI-agent-practice/blob/main/lab1_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task3：邮件助手工作流

## 1. 实验介绍

### 1.1 实验概览

在本模块中，你将探索一个**邮件助手智能体**示例。它能够像真实助理一样，使用多种工具完成复杂的邮件管理任务。

姓名：胡豪达

学号：523021910471


该智能体工作流可以执行多种邮件相关操作：发送邮件、搜索来自特定发件人的邮件、删除邮件等。你只需用自然语言下达指令——例如「查一下老板的未读邮件」或「删除 Happy Hour 那封邮件」——就能看到智能体如何选择正确的工具并完成任务。

> **名词解释：**
> - **智能体 (Agent)**：能够根据目标自主选择工具、执行步骤的 AI 系统，而不是只生成文本。
> - **工作流 (Workflow)**：一系列按顺序执行的步骤，完成一个完整任务。

<img src="https://raw.githubusercontent.com/LxRoboticsLab/AI1112-lab-assets/Lab1-2/lab_overview.png" alt="邮件助手示例" width="700"/>


### 🎯 1.2 学习目标
完成本实验后，你将能够：**将大语言模型 (LLM) 与工具连接**，用自然语言下达指令，并观察智能体如何选择、执行和验证多步任务（如搜索、发送、删除邮件）。

## 2. 环境搭建

### 前置要求

按顺序执行下方各节代码块即可完成环境搭建。

### 2.1 克隆仓库（如从 GitHub 获取）


In [ ]:
!git clone -b Lab1-2 https://github.com/LxRoboticsLab/AI1112-lab-assets.git
%cd AI1112-lab-assets

Cloning into 'AI1112-lab-assets'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 37 (delta 4), reused 29 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 1.53 MiB | 5.34 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/AI1112-lab-assets


### 2.2 安装依赖

运行下面代码块安装所需 Python 包。

In [ ]:
# @title
!pip install openai python-dotenv requests pandas uvicorn fastapi email-validator

import os
import base64

encoded_key = "c2stb3ItdjEtOTZlNzczMTQ5Y2YxODVkYzQ3ZGFiZmY3YzA0ODc4NGNiMTQ0NjUyOTJmZDIxYmEwZWUyODdhNzcwZTBhYWQwYQ=="
os.environ["OPENROUTER_API_KEY"] = base64.b64decode(encoded_key).decode() if encoded_key else ""

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 4.4 MB/s eta 0:00:00


### 2.3 配置

在下面代码块中设置邮件服务地址，然后运行。

In [ ]:
# 邮件服务地址（默认本地 8000 端口）
os.environ["M3_EMAIL_SERVER_API_URL"] = "http://localhost:8000"

### 2.5 导入依赖并初始化客户端

本实验使用 **OpenRouter** 作为 LLM 接口。

In [ ]:
# ================================
# 导入
# ================================

from dotenv import load_dotenv
import json

# 本地项目模块
import utils
import display_functions
import email_tools
from openrouter_agent import get_openrouter_client, run_agent_with_tools

# ================================
# 环境与客户端
# ================================
load_dotenv()  # 从 .env 加载环境变量
client = get_openrouter_client()  # 初始化 OpenRouter 客户端

### 2.4 启动邮件服务

运行下面代码块，在后台启动模拟邮件服务。

In [ ]:
import subprocess
import sys
import time
import requests

# 从项目根目录启动邮件服务（email_server 使用相对导入，需作为包运行）
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "email_server.email_service:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# 等待服务就绪（最多 15 秒）
url = "http://localhost:8000/emails"
for i in range(15):
    try:
        r = requests.get(url, timeout=1)
        if r.status_code == 200:
            print("✅ 邮件服务已启动，地址：http://localhost:8000")
            break
    except:
        time.sleep(1)
else:
    err = proc.stderr.read().decode() if proc.stderr else ""
    print("❌ 邮件服务启动失败。请确认：1）已运行 2.1 克隆并 cd 到项目目录；2）email_server 文件夹存在。")
    if err:
        print("错误信息：", err[:500])

✅ 邮件服务已启动，地址：http://localhost:8000


## 3. 模拟邮件服务

### 3.1 架构说明

本实验使用**模拟邮件后端**，模拟真实邮件的收发与存储，方便你在不发送真实邮件的情况下练习。

后台由以下组件构成：

| 组件 | 用途 |
|------|------|
| **FastAPI** | 提供 REST 接口，接收 HTTP 请求 |
| **SQLite + SQLAlchemy** | 本地存储与查询邮件数据 |
| **Pydantic** | 校验请求与响应的数据格式 |
| **工具层** | 连接 LLM 与邮件服务，供智能体调用 |

> **名词解释：**
> - **REST**：一种 Web API 设计风格，通过 HTTP 方法（GET/POST 等）操作资源。
> - **SQLite**：轻量级数据库，数据存放在单个文件中。

### 3.2 接口列表

服务提供以下路由（后续会被封装成工具供智能体自动调用）：

- `POST /send` → 发送邮件
- `GET /emails` → 列出所有邮件
- `GET /emails/unread` → 列出未读邮件
- `GET /emails/{id}` → 按 ID 获取指定邮件
- `GET /emails/search?q=...` → 按关键词搜索
- `GET /emails/filter` → 按收件人或日期范围筛选
- `PATCH /emails/{id}/read` → 标记为已读
- `PATCH /emails/{id}/unread` → 标记为未读
- `DELETE /emails/{id}` → 删除邮件
- `GET /reset_database` → 重置数据库（用于测试）

> 💡 **要点**：这些接口相当于「收件箱控制面板」。下一步我们会把它们封装成 Python 函数（工具），供 LLM 调用。

### 3.3 测试接口

在交给 LLM 控制之前，先**自己测试**后端是否正常工作。`utils.test_*` 函数是对这些接口的简单封装。


In [ ]:
new_email_id = utils.test_send_email()
_ = utils.test_get_email(new_email_id['id'])
_ = utils.test_list_emails()
_ = utils.test_filter_emails(recipient="test@example.com")
_ = utils.test_search_emails("lunch")
_ = utils.test_unread_emails()
_ = utils.test_mark_read(new_email_id['id'])
_ = utils.test_mark_unread(new_email_id['id'])
_ = utils.test_delete_email(new_email_id['id'])
_ = utils.reset_database()

## 4. 邮件智能体的工具层

### 4.1 为什么需要工具？

LLM 本身只会生成文字，不能直接访问数据库或 API。**工具 (Tools)** 是一组 Python 函数，把后端能力暴露给 LLM，让它能「动手」执行操作。

> 把工具想象成智能体的**执行器**：你给出自然语言指令，模型会决定**调用哪些工具**、**按什么顺序**执行。

### 4.2 工具设计原则

- **文档字符串**简明扼要，描述该工具的功能
- 返回**结构一致、紧凑的 JSON**，方便模型串联结果
- **一个工具只做一件事**，职责单一

### 4.3 可用工具列表

| 工具函数 | 功能 |
|----------|------|
| `list_all_emails()` | 获取所有邮件，按时间从新到旧 |
| `list_unread_emails()` | 获取未读邮件 |
| `search_emails(query)` | 按关键词搜索（主题、正文、发件人） |
| `filter_emails(...)` | 按收件人或日期范围筛选 |
| `get_email(email_id)` | 按 ID 获取指定邮件 |
| `mark_email_as_read(id)` | 标记为已读 |
| `mark_email_as_unread(id)` | 标记为未读 |
| `send_email(...)` | 发送一封（模拟）邮件 |
| `delete_email(id)` | 按 ID 删除邮件 |
| `search_unread_from_sender(addr)` | 获取来自指定发件人的未读邮件 |

👉 取消下面你想测试的工具行的注释，运行并查看输出。

In [ ]:
# 测试发送邮件并获取
new_email = email_tools.send_email("test@example.com", "午餐计划", "中午一起吃个饭？")
content_ = email_tools.get_email(new_email['id'])

# 取消注释你想尝试的工具：
# content_ = email_tools.list_all_emails()
content_ = email_tools.list_unread_emails()
# content_ = email_tools.search_emails("lunch")
# content_ = email_tools.filter_emails(recipient="test@example.com")
# content_ = email_tools.mark_email_as_read(new_email['id'])
# content_ = email_tools.mark_email_as_unread(new_email['id'])
# content_ = email_tools.search_unread_from_sender("test@example.com")
# content_ = email_tools.delete_email(new_email['id'])

utils.print_html(content=json.dumps(content_, indent=2, ensure_ascii=False), title="测试 email_tools")

## 5. 准备智能体提示词

在给邮件助手分配任务前，需要构建一个 `build_prompt()` 函数，把用户的自然语言请求包在系统指令中，让 LLM：

- 明确自己扮演**邮件助手智能体**
- 知道可以调用哪些工具
- **直接执行**操作，不需要向用户确认（无人工干预）

In [ ]:
def build_prompt(request_: str) -> str:
    return f"""
- 你是一个专门管理邮件的 AI 助手。
- 可以执行列出、搜索、筛选、操作邮件等操作。
- 请使用提供的工具与邮件系统交互。
- 执行操作前不需要向用户确认。
- 如需要，我的邮箱地址是 "you@email.com"，可用于发送邮件或执行与我账户相关的操作。

{request_.strip()}
"""

运行下面单元，看看 `build_prompt` 如何把你的原始指令包装成完整提示词：

In [ ]:
example_prompt = build_prompt("删除 Happy Hour 那封邮件")
utils.print_html(content=example_prompt, title="示例 prompt")

### 5.3 重置邮件服务

实验过程中可能改动过数据，建议先**重置**到初始状态：

In [ ]:
utils.reset_database()

{'message': 'Database reset and emails reloaded'}

## 6. LLM + 邮件工具

### 6.1 场景

此前你都是直接调用后端。现在让 **LLM 作为邮件助手**接管控制权。

例如你可以说：

> 「查一下 boss@email.com 的未读邮件，标记为已读，并发送一封礼貌的回复。」

### 6.2 执行流程

1. 智能体理解你的指令
2. 选择合适的工具并依次调用（`search_unread_from_sender` → `mark_email_as_read` → `send_email`）
3. 自动执行，无需你逐个确认

OpenRouter 与我们的 `openrouter_agent` 模块负责：工具定义、参数绑定、执行调用、结果回传——你只需关注**想完成什么**。

### 6.3 运行示例

👉 运行下面单元，观察智能体如何编排多个工具完成任务。你也可以修改 `prompt_` 尝试自己的指令。

*关注：*
- **工具调用链**：调用了哪些工具、传入什么参数
- **最终回复**：对执行结果做了什么总结

In [ ]:
# 可修改为你自己的请求
prompt_ = build_prompt("查一下 boss@email.com 的未读邮件，标记为已读，并发送一封礼貌的回复。")

response = run_agent_with_tools(
    client=client,
    messages=[{"role": "user", "content": prompt_}],
    tool_functions=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
    ],
    model="openai/gpt-4o-mini",  # 可通过 openrouter.ai/models 选择其他模型
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)

### 6.4 缺少工具：`delete_email`

如果智能体需要的工具**没有注册**会怎样？

例如执行：

> 「删除 alice@work.com 的邮件。」

由于 `delete_email` 不在工具列表中，LLM 会尝试处理，但**无法完成删除**。

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">
  <h4>🔍 重要结论</h4>
  <p style="margin: 0;">
    工具列表直接决定了智能体**能做什么**。没有注册的工具，智能体就无法执行对应操作。
  </p>
</div>

In [ ]:
# 尝试一个需要「未注册工具」的请求
prompt_ = build_prompt("删除 alice@work.com 的邮件")

response = run_agent_with_tools(
    client=client,
    messages=[{"role": "user", "content": prompt_}],
    tool_functions=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        # 注意：此处没有 delete_email
    ],
    model="openai/gpt-4o-mini",
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)

#### 6.4.1 启用 `delete_email` 再试一次

现在把 `delete_email` 加入工具列表，智能体就能完成删除操作。

> 提示：观察调用顺序——找到目标邮件后，智能体会调用 `delete_email` 完成删除。

In [ ]:
prompt_ = build_prompt("删除 alice@work.com 的邮件")

response = run_agent_with_tools(
    client=client,
    messages=[{"role": "user", "content": prompt_}],
    tool_functions=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email,  # 添加删除工具
    ],
    model="openai/gpt-4o-mini",
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)

### 6.5 指定操作：删除「Happy Hour」邮件

测试收件箱中预置了一封标题为 **「Happy Hour」** 的邮件。你的任务是：

> 让智能体找到并删除这封邮件。

参考数据示例：

```json
{
  "id": 1,
  "sender": "eric@work.com",
  "recipient": "you@email.com",
  "subject": "Happy Hour",
  "body": "We're planning drinks this Friday!",
  "timestamp": "2025-06-13T04:48:59.096908",
  "read": false
}
```

运行下面单元，观察智能体如何搜索并删除该邮件。

In [ ]:
prompt_ = build_prompt("删除 happy hour 那封邮件")

response = run_agent_with_tools(
    client=client,
    messages=[{"role": "user", "content": prompt_}],
    tool_functions=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email,
    ],
    model="openai/gpt-4o-mini",
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)

## 7 动手练习：组合工具实现「发送并抄送邮件」 (40%)

现有工具 `send_email(recipient, subject, body)` 每次只能指定**一个收件人**。请你在下方代码框中，**基于现有工具进行组合**，实现一个新工具：**发送并抄送邮件**。

**要求：**
- 新工具应支持：主收件人 `recipient`、抄送列表 `cc_list`（可多个人）、主题 `subject`、正文 `body`。
- 实现方式：复用已有的 `email_tools.send_email()`，对主收件人调用一次，对抄送列表中的每个地址再各调用一次（同一封邮件内容发多份，模拟「发件人 + 抄送」效果）。
- 为新函数写清**文档字符串**，说明参数与返回值，便于后续作为智能体工具注册使用。

**预期效果：** 用户提出要发送并抄送邮件后，LLM能正确调用新的工具`send_email(recipient, subject, body)`来完成任务，而不是使用之前的`email_tools.send_email()`。

👉 在下方单元格中实现你的 `send_email_with_cc`，并做一次简单调用测试。

In [ ]:
def send_email_with_cc(recipient: str, cc_list: list, subject: str, body: str):
    """
    发送邮件并抄送给 cc_list 中的收件人。
    组合现有 send_email：先发给主收件人，再对每位抄送人各发一封相同主题与正文的邮件。

    Args:
        recipient (str): 主收件人邮箱地址。
        cc_list (list): 抄送人邮箱地址列表，如 ["cc1@example.com", "cc2@example.com"]。
        subject (str): 邮件主题。
        body (str): 邮件正文。

    Returns:
        list: 每封已发送邮件对应的记录（或汇总信息），便于调用方检查。
    """
    results = []
    # write your code here
    results.append(email_tools.send_email(recipient, subject, body))
    for i in range(len(cc_list)):
        results.append(email_tools.send_email(cc_list[i], subject, body))
    return results


# 简单测试：发一封邮件并抄送给两人
result = send_email_with_cc("main@example.com", ["cc1@example.com", "cc2@example.com"], "会议提醒", "明天下午 3 点开会。")
utils.print_html(content=json.dumps(result, indent=2, ensure_ascii=False), title="发送并抄送结果")

#### 7.1 让 LLM 使用「发送并抄送」工具完成任务

实现好 `send_email_with_cc` 后，请进一步将其**加入智能体的工具列表**，让 LLM 根据自然语言指令自动选择并调用该工具。

**补全以下tool description**

👉 运行下面单元：用一句自然语言（例如「给 team@company.com 发一封主题为『周报提醒』的邮件，正文写『请在本周五前提交周报』，并抄送给 boss@company.com 和 hr@company.com」），观察智能体是否会调用你实现的 `send_email_with_cc` 完成任务。

In [ ]:
# 使用自然语言让智能体「发送并抄送」邮件（请确保上方已实现 send_email_with_cc）
import openrouter_agent
_orig_build = openrouter_agent.build_tool_definitions
def _patched_build_tool_definitions(tool_functions):
    tools, name_to_func = _orig_build(tool_functions)
    if "send_email_with_cc" in name_to_func and not any(t.get("function", {}).get("name") == "send_email_with_cc" for t in tools):
        tools.append({
            "type": "function",
            "function": {
                "name": "send_email_with_cc",
                "description": "发送邮件给主收件人并抄送给多个抄送人。",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "recipient": {"type": "string", "description": "主收件人邮箱地址"},
                        "cc_list": {"type": "array", "items": {"type": "string"}, "description": "抄送人邮箱地址列表"},
                        "subject": {"type": "string", "description": "邮件主题"},
                        "body": {"type": "string", "description": "邮件正文"}
                    },
                    "required": ["recipient", "cc_list", "subject", "body"],
                },
            },
        })
    return tools, name_to_func
openrouter_agent.build_tool_definitions = _patched_build_tool_definitions

prompt_ = build_prompt(
    "给 team@company.com 发一封主题为「周报提醒」的邮件，"
    "正文写「请在本周五前提交周报」，并抄送给 boss@company.com 和 hr@company.com。"
)

response = run_agent_with_tools(
    client=client,
    messages=[{"role": "user", "content": prompt_}],
    tool_functions=[
        # email_tools.search_unread_from_sender,
        # email_tools.list_unread_emails,
        # email_tools.search_emails,
        # email_tools.get_email,
        # email_tools.mark_email_as_read,
        email_tools.send_email,
        send_email_with_cc,  # 使用你实现的「发送并抄送」工具
        # email_tools.delete_email,
    ],
    model="openai/gpt-4o-mini",
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)

## 8. 总结

- 本实验展示了 LLM 邮件智能体如何通过**工具调用**与模拟邮件服务交互。
- **工具调用**使 LLM 不仅能生成文本，还能执行函数、完成多步任务。
- **可用工具集合**决定了智能体的能力（例如没有 `delete_email` 就无法删除邮件）。
- 清晰的文档字符串和 consistent（一致）的行为有助于 LLM 选择正确的工具。
- OpenRouter + 我们的工具循环模块负责：工具暴露、参数绑定、执行与结果回传。
- 观察完整流程——从提示词 → 工具调用 → 输出 → 最终回复——是理解并改进智能体推理与行为的关键。

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>

你已引导一个由 LLM 驱动的<em>邮件助手智能体</em>，从自然语言请求走向模拟邮件服务上的具体操作。你看到了工具如何把自然语言转变为可靠操作——列出、搜索、标为已读、发送、删除——而你无需直接发 HTTP 请求。

你了解到：暴露给智能体的工具决定了它的实际能力。工具缺失时，智能体只能推理无法执行；工具齐全时，它可以串联步骤、传递参数，按你的意图完成任务。在此过程中，OpenRouter 和我们的工具模块负责底层对接，让你专注于「要做什么」，而不是「怎么连 API」。

有了这份工作流，你可以开始设计面向任务的智能体：安全、透明、可解释，从简单 prompt 扩展到可靠的、跨服务的多步自动化。 🌟
</div>